# Domain_Behaviour — how does each domain behave in this COSMoS release?

Re-derives, from the published COSMoS content, the behaviour that the Findings
consumers' scope decisions rest on, and reports where content and decisions disagree.
It reports only. It never changes a scope or a classification; those stay in
`SDTM_Domain_Metadata.xlsx`.

This is the March 2026 behavioural analysis
(`docs/archive/behavioural-analysis-2026-03/`) made reproducible: run it on every
COSMoS release and compare with the previous run.

**Inputs**
- `cosmos-graph/interim/COSMoS_Graph.xlsx` — DSS, BC, Variables and DataElementConcepts sheets
- `consumer-bases/interim/DSS_View.xlsx` — `Measurement_Specs` (one row per DSS, variables pivoted)
- `sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx` — `Domains`,
  `Instrument_Domain_Rules`, `Consumer_Exclusions`, `Variable_Meaning`
- `cosmos-bc-dss/reports/COSMoS_Observable_Derivation.xlsx` — `DSS_Coordinates` (the observable key per DSS)
- `sdtm-findings-graph/machine_actionable/Specimen_Findings.xlsx` — `Measurement_Specs` (the consumer's observable key, LB, MB, MI)
- `sdtm-test-codes/downloads/SDTM_Terminology.txt` — codelist names and definitions
- `sdtm-domain-reference/downloads/SDTM_v2.0.csv`, `SDTMIG_v3.4.csv` — CDISC variable tables (role, qualified variables, CDISC Notes); gitignored
- `sdtm-domain-reference/downloads/SDTMIG v3.4-FINAL_2022-07-21.pdf` — the SDTMIG document, for the domain Assumptions; gitignored; read with pypdf (needs `cryptography`)

**Output** — `domain-behaviour/interim/Domain_Behaviour.xlsx`

| Sheet | Grain | Content |
|---|---|---|
| `Domain_Profile` | domain | Size, fan-out, primary axis, scale mix, variable shares; metadata class and consumer status |
| `Fanout_Axes` | BC with more than one DSS | Which axes distinguish its DSSs |
| `Key_Compare` | BC with more than one DSS | The fan-out axes against the observable key of `COSMoS_Observable_Derivation` |
| `Variable_Evidence` | variable × domain | What COSMoS, SDTM CT, the SDTM v2.0 Model and SDTMIG v3.4 publish about each variable that differs between the DSSs of one BC |
| `Consumer_Key_Compare` | key group that differs | The observable key of `Specimen_Findings` against the derivation's key, for the same DSSs |
| `IG_Assumptions` | numbered SDTMIG v3.4 domain assumption | Assumptions in our Findings domains that name one of our variables, quoted as extracted |
| `LOINC_Compare` | BC with more than one DSS and a LOINC code | The fan-out axes against the LOINC axes that differ between its DSSs |
| `LBCAT_LOINC_Class` | LBCAT × LOINC class | LB DSSs with a LOINC code, by assigned category and LOINC class |
| `Rater_Evidence` | instrument codelist | CT class and the phrases in the NCIt definition that name who completes or rates it |
| `Rater_COSMoS_EVAL` | `--EVAL` / `--EVALID` × assigned value | DSSs and BCs in QS, FT, RS |
| `Rater_Class_Moves` | instrument whose class changed | Class in the prior and current SDTM CT, with the rater phrases in its NCIt definition |
| `Scale_Units` | domain × published result-scale combination | DSSs with and without units; flags |
| `Exclusion_Claims` | claim in `Consumer_Exclusions` | Holds / does not hold, with the evidence |
| `Unclassified_Content` | domain | Domains with DSSs but no consumer class |
| `Key_Measures`, `Pins` | — | Measures for the release-to-release diff; input versions |

## 1. Setup

In [1]:
from pathlib import Path
from collections import Counter
import csv
import re

import pandas as pd

REPO = Path.cwd().parent.parent  # notebooks/ -> domain-behaviour/ -> repo root
GRAPH = REPO / 'cosmos-graph' / 'interim' / 'COSMoS_Graph.xlsx'
DSS_VIEW = REPO / 'consumer-bases' / 'interim' / 'DSS_View.xlsx'
DOMAIN_META = REPO / 'sdtm-domain-reference' / 'machine_actionable' / 'SDTM_Domain_Metadata.xlsx'
OBS_KEY = REPO / 'cosmos-bc-dss' / 'reports' / 'COSMoS_Observable_Derivation.xlsx'
LOINC_CHECK = REPO / 'cosmos-bc-dss' / 'reports' / 'COSMoS_Observable_LOINC_Check.xlsx'
SPECIMEN_FINDINGS = REPO / 'sdtm-findings-graph' / 'machine_actionable' / 'Specimen_Findings.xlsx'
CT_FILE = REPO / 'sdtm-test-codes' / 'downloads' / 'SDTM_Terminology.txt'
INSTRUMENT_ID = REPO / 'sdtm-test-codes' / 'machine_actionable' / 'SDTM_Instrument_Identity.xlsx'
THESAURUS = REPO / 'sdtm-test-codes' / 'downloads' / 'Thesaurus.txt'
# Prior SDTM CT for the instrument class moves in section 4e; None skips them.
CT_PRIOR_FILE = REPO / 'sdtm-test-codes' / 'downloads' / '_prev_2026-03-27' / 'SDTM_Terminology.txt'
OUT = REPO / 'domain-behaviour' / 'interim' / 'Domain_Behaviour.xlsx'
# CDISC variable tables (gitignored; downloaded with a cdisc.org sign-in)
SDTM_MODEL = REPO / 'sdtm-domain-reference' / 'downloads' / 'SDTM_v2.0.csv'
SDTMIG = REPO / 'sdtm-domain-reference' / 'downloads' / 'SDTMIG_v3.4.csv'
NSV_REGISTRY = REPO / 'sdtm-domain-reference' / 'downloads' / 'Approved-Non-Standard-Variable-Registry_2026-04-03.xlsx'
IG_PDF = REPO / 'sdtm-domain-reference' / 'downloads' / 'SDTMIG v3.4-FINAL_2022-07-21.pdf'

# Optional: path to a previous Domain_Behaviour.xlsx for the release diff in the final cell.
PRIOR_FILE = None

for path in [GRAPH, DSS_VIEW, DOMAIN_META, OBS_KEY, LOINC_CHECK, SPECIMEN_FINDINGS, CT_FILE, INSTRUMENT_ID, THESAURUS, SDTM_MODEL, SDTMIG, NSV_REGISTRY, IG_PDF]:
    if not path.exists():
        raise FileNotFoundError(f'{path} not found')
if CT_PRIOR_FILE and not CT_PRIOR_FILE.exists():
    raise FileNotFoundError(f'{CT_PRIOR_FILE} not found')
OUT.parent.mkdir(exist_ok=True)
print('repo:', REPO)

repo: /Users/kerstinforsberg/repos/GitHub/cdisc-for-ai


## 2. Load

In [2]:
dss = pd.read_excel(GRAPH, sheet_name='DSS', dtype=str).fillna('')
bc = pd.read_excel(GRAPH, sheet_name='BC', dtype=str).fillna('')
view = pd.read_excel(DSS_VIEW, sheet_name='Measurement_Specs', dtype=str).fillna('')
meta = pd.read_excel(DOMAIN_META, sheet_name='Domains', dtype=str).fillna('')
instr_rules = pd.read_excel(DOMAIN_META, sheet_name='Instrument_Domain_Rules', dtype=str).fillna('')
excl = pd.read_excel(DOMAIN_META, sheet_name='Consumer_Exclusions', dtype=str).fillna('')
meaning = pd.read_excel(DOMAIN_META, sheet_name='Variable_Meaning', dtype=str).fillna('')

if len(view) != len(dss):
    raise ValueError(f'DSS_View has {len(view)} DSS rows, COSMoS_Graph DSS sheet has {len(dss)} - rebuild consumer-bases first')

package_date = dss['package_date'].max()
print(f'COSMoS package_date (latest): {package_date}')
print(f'DSSs: {len(dss):,}  BCs with DSS: {dss["bc_id"].nunique():,}  domains: {dss["domain"].nunique()}')

COSMoS package_date (latest): 2026-07-14
DSSs: 1,475  BCs with DSS: 1,008  domains: 32


## 3. Axes

An axis is a variable whose value can differ between the DSSs of one BC. It is counted
when the assigned value differs. Where only the value list differs, the axis is reported
separately (`Axes_Value_List_Only`). Units, codelists,
LOINC codes and result values follow from the axes and are not counted as axes.
`scale` is read from units: some DSSs of the BC carry units, others do not.

In [3]:
AXIS_VARIABLES = {
    'specimen': ['SPEC'],
    'method': ['METHOD', 'ANMETH'],
    'binding agent': ['BDAGNT'],
    'test detail': ['TSTDTL'],
    'location': ['LOC', 'LAT', 'POS', 'DIR', 'PORTOT'],
    'category': ['CAT', 'SCAT'],
    'evaluator': ['EVAL', 'EVALID'],
}
AXIS_COLUMNS = {
    axis: [f'{v}{s}' for v in variables for s in ('_value', '_value_list') if f'{v}{s}' in view.columns]
    for axis, variables in AXIS_VARIABLES.items()
}
UNIT_COLUMNS = [c for c in view.columns if c.startswith('ORRESU_')]
view['has_units'] = view[UNIT_COLUMNS].ne('').any(axis=1)

for axis, cols in AXIS_COLUMNS.items():
    print(f'  {axis:14} {cols}')

  specimen       ['SPEC_value', 'SPEC_value_list']
  method         ['METHOD_value', 'METHOD_value_list', 'ANMETH_value', 'ANMETH_value_list']
  binding agent  ['BDAGNT_value']
  test detail    ['TSTDTL_value']
  location       ['LOC_value', 'LOC_value_list', 'LAT_value_list', 'POS_value_list', 'DIR_value_list']
  category       ['CAT_value', 'CAT_value_list', 'SCAT_value', 'SCAT_value_list']
  evaluator      ['EVAL_value', 'EVAL_value_list', 'EVALID_value_list']


## 4. Fan-out axes per BC

In [4]:
rows = []
for (domain, bc_id), g in view.groupby(['domain', 'bc_id']):
    if len(g) < 2:
        continue
    axes = [axis for axis, cols in AXIS_COLUMNS.items() if any(g[c].nunique() > 1 for c in cols if c.endswith('_value'))]
    list_only = [axis for axis, cols in AXIS_COLUMNS.items()
                 if axis not in axes and any(g[c].nunique() > 1 for c in cols if c.endswith('_value_list'))]
    if g['has_units'].nunique() > 1:
        axes.append('scale')
    rows.append({
        'domain': domain,
        'bc_id': bc_id,
        'bc_short_name': g['bc_short_name'].iloc[0],
        'DSS_Count': len(g),
        'Axes': '; '.join(axes) if axes else '(none)',
        'Axes_Value_List_Only': '; '.join(list_only),
        'DS_Codes': '; '.join(sorted(g['ds_id'])),
    })
fanout = pd.DataFrame(rows).sort_values(['domain', 'bc_id']).reset_index(drop=True)
print(f'BCs with more than one DSS: {len(fanout)}')

BCs with more than one DSS: 83


## 4b. Fan-out axes against the observable key

`COSMoS_Observable_Derivation` gives every DSS with a `--ORRES` variable (QS excluded) a key:
`bc_id | component | system | scale | method`. This section compares, per BC that fans
out, what separates its DSSs here with what separates them in the key. It reports the
difference. It changes neither derivation.

The two read scale differently: here from units (some DSSs carry units, others do not),
in the key from the data type of `--ORRES`. COSMoS publishes result scales per BC, not per DSS.

In [5]:
coord = pd.read_excel(OBS_KEY, sheet_name='DSS_Coordinates', dtype=str).fillna('')
key_readme = pd.read_excel(OBS_KEY, sheet_name='README', header=None, dtype=str).fillna('')[0]
key_source = key_readme[key_readme.str.startswith('Source:')].iloc[0]
if package_date not in key_source:
    raise ValueError(f'COSMoS_Observable_Derivation.xlsx is not from package {package_date}: {key_source}')
unknown = set(coord['ds_id']) - set(view['ds_id'])
if unknown:
    raise ValueError(f'DSSs in the observable key but not in DSS_View: {sorted(unknown)}')

# The key is read as published: bc_id | component | system | scale | method.
coord[['key_system', 'key_scale', 'key_method']] = coord['observable_key'].str.rsplit('|', n=3, expand=True)[[1, 2, 3]]
kc = view[['ds_id', 'domain', 'bc_id', 'TESTCD_value', 'BDAGNT_value', 'TSTDTL_value', 'OBJ_value']].merge(
    coord[['ds_id', 'observable_key', 'component', 'key_system', 'key_scale', 'key_method']], on='ds_id', how='left').fillna('')

# Fan-out axis -> the part of the key that should carry it. Category, location and evaluator have none.
AXIS_TO_KEY = {'specimen': 'system', 'binding agent': 'binding agent', 'test detail': 'test detail', 'method': 'method', 'scale': 'scale'}
KEY_PARTS = {'system': 'key_system', 'scale': 'key_scale', 'method': 'key_method', 'test code': 'TESTCD_value',
             'binding agent': 'BDAGNT_value', 'test detail': 'TSTDTL_value', 'object': 'OBJ_value'}
COMPONENT_PARTS = ['test code', 'binding agent', 'test detail', 'object']

rows = []
for r in fanout.itertuples(index=False):
    g = kc[(kc['domain'] == r.domain) & (kc['bc_id'] == r.bc_id)]
    in_key = int(g['observable_key'].ne('').sum())
    axes = [] if r.Axes == '(none)' else r.Axes.split('; ')
    row = {'domain': r.domain, 'bc_id': r.bc_id, 'bc_short_name': r.bc_short_name, 'DSS_Count': r.DSS_Count,
           'In_Key_Scope': 'all' if in_key == len(g) else 'none' if in_key == 0 else 'part',
           'Observable_Keys': '', 'Axes': r.Axes, 'Axes_Value_List_Only': r.Axes_Value_List_Only,
           'Key_Differs_By': '', 'Both': '', 'Fanout_Only': '', 'Key_Only': ''}
    if row['In_Key_Scope'] == 'part':
        raise ValueError(f'{r.domain} {r.bc_id}: only {in_key} of {len(g)} DSSs are in the observable key')
    if row['In_Key_Scope'] == 'all':
        differs = [part for part, col in KEY_PARTS.items() if g[col].nunique() > 1]
        if (g['component'].nunique() > 1) != any(p in differs for p in COMPONENT_PARTS):
            raise ValueError(f'{r.domain} {r.bc_id}: key component does not follow TESTCD, BDAGNT, TSTDTL, OBJ')
        mapped = {AXIS_TO_KEY[a] for a in axes if a in AXIS_TO_KEY}
        row['Observable_Keys'] = g['observable_key'].nunique()
        row['Key_Differs_By'] = '; '.join(differs) if differs else '(none)'
        row['Both'] = '; '.join(a for a in axes if AXIS_TO_KEY.get(a) in differs)
        row['Fanout_Only'] = '; '.join(a for a in axes if AXIS_TO_KEY.get(a) not in differs)
        row['Key_Only'] = '; '.join(p for p in differs if p not in mapped)
    rows.append(row)
key_compare = pd.DataFrame(rows)

in_scope = key_compare[key_compare['In_Key_Scope'] == 'all']
print(f'BCs that fan out: {len(key_compare)}   in the scope of the observable key: {len(in_scope)}')
print('Not in scope, by domain:', key_compare.loc[key_compare['In_Key_Scope'] == 'none', 'domain'].value_counts().sort_index().to_dict())
print()
print(f'{"":16}{"both":>6}{"fan-out only":>14}{"key only":>10}')
key_agreement = []
for label in list(AXIS_TO_KEY) + ['category', 'location', 'evaluator', 'test code', 'object']:
    part = AXIS_TO_KEY.get(label, label)
    counts = [int(in_scope[c].str.split('; ').map(lambda v: x in v).sum()) for c, x in (('Both', label), ('Fanout_Only', label), ('Key_Only', part))]
    key_agreement.append((label, *counts))
    print(f'{label:16}{counts[0]:>6}{counts[1]:>14}{counts[2]:>10}')
keys = in_scope['Observable_Keys'].astype(int)
key_grain = {'one key per DSS': int((keys == in_scope['DSS_Count']).sum()), 'one key for all DSSs': int((keys == 1).sum())}
key_grain['in between'] = len(in_scope) - sum(key_grain.values())
print()
print('Keys against DSSs:', key_grain)

BCs that fan out: 83   in the scope of the observable key: 70
Not in scope, by domain: {'AE': 1, 'CM': 1, 'EC': 1, 'MH': 1, 'PR': 5, 'SU': 3, 'TS': 1}

                  both  fan-out only  key only
specimen            29             0         0
binding agent        6             0         0
test detail         18             0         0
method               8             1         0
scale               17             3         7
category             0            12         0
location             0             0         0
evaluator            0             0         0
test code            0             0         1
object               0             0         0

Keys against DSSs: {'one key per DSS': 49, 'one key for all DSSs': 16, 'in between': 5}


## 4c. What the standards publish about each variable

One row per variable and domain where, in at least one BC that fans out, the assigned
value or the value list differs between the DSSs. All variables are read, not only the
seven axes. Variables that follow from the axes are left out: test name, result, units, LOINC.

For each row the sheet quotes what is published: the COSMoS linking phrase, predicate
term and data element concept label, and the SDTM CT codelist name and definition.
`(not stated)` means the source is empty. Nothing here is classified by us.

`Generic_Variable` is the SDTM `--` name of the variable (`LBCAT` → `--CAT`).
`Published_Differs_In` lists the other domains where the same generic variable has a
different COSMoS predicate or linking phrase, with what is published there. It records
that the statements differ. It does not say whether the difference is an inconsistency
or follows from the domain class.

`Observation_Class` comes from `SDTM_Domain_Metadata.xlsx`. Phase 1 reads the Findings
rows; Events and Interventions rows stay in the sheet for later.

The SDTM v2.0 Model and SDTMIG v3.4 variable tables add what the data standards themselves say: `SDTM_Role`
and `SDTM_Variables_Qualified` (Model, generic variable, by observation class), `SDTM_Definition`, `SDTM_Notes`,
and `IG_Role`, `IG_Core`, `IG_CDISC_Notes` (the domain's own variable in the IG). `(not in SDTM v2.0)` and
`(not in SDTMIG v3.4)` mean the variable is not in that table. `IG_Role_Differs_From_Model` records where the two disagree.
`COSMoS_Nonstandard` is the COSMoS flag `is_nonstandard` (Y/N), as published. `NSV_Registry` quotes the CDISC
Approved Non-Standard Variable Registry entry for the variable fragment, only where the variable is not in the
domain's SDTMIG v3.4 table.

In [6]:
variables = pd.read_excel(GRAPH, sheet_name='Variables', dtype=str).fillna('')
decs = pd.read_excel(GRAPH, sheet_name='DataElementConcepts', dtype=str).fillna('')
ct = pd.read_csv(CT_FILE, sep='\t', dtype=str).fillna('')
codelists = ct[ct['Codelist Code'] == ''].set_index('Code')  # codelist-level rows

variables['suffix'] = variables['variable_name'].str[2:]
variables = variables.merge(decs[['bc_id', 'dec_id', 'dec_label']].drop_duplicates(['bc_id', 'dec_id']), on=['bc_id', 'dec_id'], how='left').fillna('')

# Not listed: variables that follow from the axes (see section 3).
FOLLOWS = {'TEST', 'ORRES', 'ORRESU', 'STRESC', 'STRESN', 'STRESU', 'LOINC'}
AXIS_OF = {v: axis for axis, vs in AXIS_VARIABLES.items() for v in vs}
KEY_PART_OF = {'TESTCD': 'component', 'BDAGNT': 'component', 'TSTDTL': 'component', 'OBJ': 'component',
               'SPEC': 'system', 'LOC': 'system, when no specimen is assigned', 'METHOD': 'method'}
key_domains = set(in_scope['domain'])
suffixes = sorted({c.rsplit('_value', 1)[0] for c in view.columns if c.endswith(('_value', '_value_list'))} - FOLLOWS)

def stated(values):
    found = sorted({v for v in values if v})
    if not found:
        return '(not stated)'
    return ' | '.join(found) + (' | (not stated)' if any(v == '' for v in values) else '')

fan_view = view.merge(fanout[['domain', 'bc_id']], on=['domain', 'bc_id'])
rows = []
for domain, d in fan_view.groupby('domain'):
    for suffix in suffixes:
        value_col, list_col = f'{suffix}_value', f'{suffix}_value_list'
        assigned_bcs = [b for b, g in d.groupby('bc_id') if value_col in d.columns and g[value_col].nunique() > 1]
        list_bcs = [b for b, g in d.groupby('bc_id') if b not in assigned_bcs and list_col in d.columns and g[list_col].nunique() > 1]
        if not assigned_bcs and not list_bcs:
            continue
        a = d[d['bc_id'].isin(assigned_bcs)]
        terms = a.loc[a[value_col] != '', [value_col, f'{suffix}_ncit']].drop_duplicates() if assigned_bcs else pd.DataFrame()
        v = variables[variables['ds_id'].isin(d['ds_id']) & (variables['suffix'] == suffix)]
        codelist_codes = sorted(set(v['codelist_concept_id']) - {''})
        unknown = [c for c in codelist_codes if c not in codelists.index]
        if unknown:
            raise ValueError(f'{domain} {suffix}: codelist {unknown} not in {CT_FILE.name}')
        rows.append({
            'domain': domain,
            'variable': '; '.join(sorted(set(v['variable_name']))),
            'Generic_Variable': f'--{suffix}',
            'Axis': AXIS_OF.get(suffix, ''),
            'In_Observable_Key': KEY_PART_OF.get(suffix, 'no') if domain in key_domains else 'domain not in the scope of the key',
            'BCs_Assigned_Differs': len(assigned_bcs),
            'BCs_Value_List_Only': len(list_bcs),
            'Assigned_Values': '' if not assigned_bcs else '; '.join(sorted(terms[value_col])) if len(terms) <= 12 else f'{len(terms)} values',
            'Values_With_NCIt': '' if not assigned_bcs else f'{int((terms[f"{suffix}_ncit"] != "").sum())} of {len(terms)}',
            'Codelist': '; '.join(codelists.loc[c, 'CDISC Submission Value'] for c in codelist_codes),
            'Codelist_Code': '; '.join(codelist_codes),
            'Codelist_Name': ' | '.join(codelists.loc[c, 'Codelist Name'] for c in codelist_codes),
            'Codelist_Definition': ' | '.join(codelists.loc[c, 'CDISC Definition'] for c in codelist_codes),
            'COSMoS_Linking_Phrase': stated(v['linking_phrase']),
            'COSMoS_Predicate': stated(v['predicate_term']),
            'COSMoS_DEC_Label': stated(v['dec_label']),
        })
variable_evidence = pd.DataFrame(rows)

# Same generic variable, published differently in another domain (predicate or linking phrase).
statement = variable_evidence['COSMoS_Predicate'] + ' / ' + variable_evidence['COSMoS_Linking_Phrase']

def differs_in(i):
    r = variable_evidence.loc[i]
    other = variable_evidence[(variable_evidence['Generic_Variable'] == r['Generic_Variable'])
                              & (variable_evidence['domain'] != r['domain'])
                              & (statement != statement[i])]
    return '; '.join(f'{d}: {s}' for d, s in zip(other['domain'], statement[other.index]))

variable_evidence['Published_Differs_In'] = [differs_in(i) for i in variable_evidence.index]

domain_class = meta.set_index('Domain')['Observation_Class']
missing = sorted(set(variable_evidence['domain']) - set(domain_class.index))
if missing:
    raise ValueError(f'Domains not in {DOMAIN_META.name}: {missing}')
variable_evidence.insert(1, 'Observation_Class', variable_evidence['domain'].map(domain_class))

# SDTM v2.0 Model (generic variables, by class) and SDTMIG v3.4 (domain variables), as published.
sdtm_model = pd.read_csv(SDTM_MODEL, dtype=str).fillna('')
sdtmig = pd.read_csv(SDTMIG, dtype=str).fillna('')
generic = sdtm_model[sdtm_model['Dataset Name'] == ''].set_index(['Class', 'Variable Name'])
if generic.index.duplicated().any():
    raise ValueError(f'{SDTM_MODEL.name}: more than one row for a class and generic variable')
ig = sdtmig.set_index(['Dataset Name', 'Variable Name'])
if ig.index.duplicated().any():
    raise ValueError(f'{SDTMIG.name}: more than one row for a domain and variable')


def model_row(cls, generic_variable):
    for c in (cls, 'General Observations'):
        if (c, generic_variable) in generic.index:
            return generic.loc[(c, generic_variable)]
    return None


def ig_field(domain, names, field):
    return ' | '.join(dict.fromkeys(ig.loc[(domain, v), field] if (domain, v) in ig.index else '(not in SDTMIG v3.4)'
                                    for v in names.split('; ')))


model = [model_row(c, g) for c, g in zip(variable_evidence['Observation_Class'], variable_evidence['Generic_Variable'])]
variable_evidence['SDTM_Role'] = [m['Role'] if m is not None else '(not in SDTM v2.0)' for m in model]
variable_evidence['SDTM_Variables_Qualified'] = [m['Variables Qualified'] if m is not None else '' for m in model]
variable_evidence['SDTM_Definition'] = [stated([m['Definition']]) if m is not None else '' for m in model]
variable_evidence['SDTM_Notes'] = [stated([m['Notes']]) if m is not None else '' for m in model]
variable_evidence['IG_Role'] = [ig_field(d, v, 'Role') for d, v in zip(variable_evidence['domain'], variable_evidence['variable'])]
variable_evidence['IG_Core'] = [ig_field(d, v, 'Core') for d, v in zip(variable_evidence['domain'], variable_evidence['variable'])]
variable_evidence['IG_CDISC_Notes'] = [ig_field(d, v, 'CDISC Notes') for d, v in zip(variable_evidence['domain'], variable_evidence['variable'])]
absent = ('(not in SDTM v2.0)', '(not in SDTMIG v3.4)')
variable_evidence['IG_Role_Differs_From_Model'] = [
    'yes' if s not in absent and i not in absent and s != i else ''
    for s, i in zip(variable_evidence['SDTM_Role'], variable_evidence['IG_Role'])]

# COSMoS's own flag, and the CDISC Approved Non-Standard Variable Registry for variables not in the domain's SDTMIG v3.4 table.
# The registry lists variable fragments (no domain prefix); entries are quoted as found.
nonstandard = variables.groupby('variable_name')['is_nonstandard'].agg(lambda s: '; '.join(sorted(set(s) - {''})))
variable_evidence['COSMoS_Nonstandard'] = [' | '.join(dict.fromkeys(nonstandard.get(v, '') for v in names.split('; ')))
                                         for names in variable_evidence['variable']]
nsv_raw = pd.read_excel(NSV_REGISTRY, sheet_name='Approved NSVs-alpha', header=None, dtype=str).fillna('')
if nsv_raw.iloc[2, 0] != 'Variable Name':
    raise ValueError(f'{NSV_REGISTRY.name}: header row not where expected')
nsv = nsv_raw.iloc[3:].set_axis(list(nsv_raw.iloc[2]), axis=1)
nsv = nsv[nsv['Variable Name'].str.strip() != ''].assign(fragment=lambda d: d['Variable Name'].str.strip())
nsv_text = {f: ' | '.join(dict.fromkeys(f"{r['Status of the NSV Review']}; {r['Role']}; used in {r['Used in Domain(s)'] or '(not stated)'}; {r['Source(s)']}"
                                        for _, r in g.iterrows()))
            for f, g in nsv.groupby('fragment')}
variable_evidence['NSV_Registry'] = [nsv_text.get(g[2:], '(not in the registry)') if i == '(not in SDTMIG v3.4)' else ''
                                     for g, i in zip(variable_evidence['Generic_Variable'], variable_evidence['IG_Role'])]
findings = variable_evidence[variable_evidence['Observation_Class'] == 'Findings']

n_assigned = int((variable_evidence['BCs_Assigned_Differs'] > 0).sum())
print(f'Variable x domain pairs: {len(variable_evidence)}   assigned value differs: {n_assigned}   only the value list differs: {len(variable_evidence) - n_assigned}')
print('Without a codelist:', int((variable_evidence['Codelist_Code'] == '').sum()),
      '  without a linking phrase:', int((variable_evidence['COSMoS_Linking_Phrase'] == '(not stated)').sum()),
      '  without a DEC label:', int((variable_evidence['COSMoS_DEC_Label'] == '(not stated)').sum()))
multi = variable_evidence.groupby('Generic_Variable')['domain'].nunique()
multi = multi[multi > 1].index
differ = sorted(variable_evidence.loc[variable_evidence['Published_Differs_In'] != '', 'Generic_Variable'].unique())
print(f'Generic variables in more than one domain: {len(multi)}   published differently across domains: {len(differ)} ({", ".join(differ)})')
n_assigned_findings = int((findings['BCs_Assigned_Differs'] > 0).sum())
print(f'Findings: pairs {len(findings)}   assigned value differs: {n_assigned_findings}   only the value list differs: {len(findings) - n_assigned_findings}')
print('Findings: without a codelist:', int((findings['Codelist_Code'] == '').sum()),
      '  without a linking phrase:', int((findings['COSMoS_Linking_Phrase'] == '(not stated)').sum()),
      '  without a DEC label:', int((findings['COSMoS_DEC_Label'] == '(not stated)').sum()),
      '  published differently from another domain:', ', '.join(findings.loc[findings['Published_Differs_In'] != '', 'variable']))
print('Findings: not in SDTMIG v3.4:', ', '.join(findings.loc[findings['IG_Role'] == '(not in SDTMIG v3.4)', 'variable']) or '(none)',
      '  IG role differs from the Model:', ', '.join(findings.loc[findings['IG_Role_Differs_From_Model'] == 'yes', 'variable']) or '(none)')
print('Findings: COSMoS non-standard:', ', '.join(findings.loc[findings['COSMoS_Nonstandard'].str.contains('Y'), 'variable']) or '(none)')

Variable x domain pairs: 74   assigned value differs: 40   only the value list differs: 34
Without a codelist: 25   without a linking phrase: 18   without a DEC label: 14
Generic variables in more than one domain: 17   published differently across domains: 6 (--CAT, --GRPID, --INDC, --LOC, --SCAT, --TRT)
Findings: pairs 27   assigned value differs: 18   only the value list differs: 9
Findings: without a codelist: 4   without a linking phrase: 1   without a DEC label: 5   published differently from another domain: LBCAT, MKLOC, RSCAT, TRGRPID
Findings: not in SDTMIG v3.4: TRREASNE   IG role differs from the Model: MITSTDTL
Findings: COSMoS non-standard: TRREASNE


## 4d. LOINC as a second source

`COSMoS_Observable_LOINC_Check` gives LOINC's own axes (component, property, time, system,
scale, method, class) for each LOINC code a DSS pins. Only DSSs with a LOINC code are
covered. For each BC that fans out and has DSSs with a LOINC code, `LOINC_Compare` sets
the COSMoS fan-out axes beside the LOINC axes whose values differ between those DSSs.
A DSS that pins several codes shows all their values. Where LOINC and COSMoS differ, it is
stated, not resolved.

The counts pair specimen with system, method with method and scale with scale, as the
LOINC check does. The COSMoS scale axis is read from units, LOINC scale is LOINC's own
(Qn, Ord, Nom). Component, property, time and class have no COSMoS axis and are counted
on their own.

`LBCAT_LOINC_Class`: every LB DSS with a LOINC code, its assigned `LBCAT` against the
LOINC class of its codes. One count per DSS and class.

In [7]:
loinc = pd.read_excel(LOINC_CHECK, sheet_name='Axis_Compare', dtype=str).fillna('')
loinc_readme = pd.read_excel(LOINC_CHECK, sheet_name='README', header=None, dtype=str).fillna('')[0]
loinc_generated = loinc_readme[loinc_readme.str.startswith('Generated:')].iloc[0]
key_generated = key_readme[key_readme.str.startswith('Generated:')].iloc[0]
if loinc_generated < key_generated:
    raise ValueError(f'COSMoS_Observable_LOINC_Check.xlsx ({loinc_generated}) is older than COSMoS_Observable_Derivation.xlsx ({key_generated}); re-run the LOINC check')
unknown = set(loinc['ds_id']) - set(view['ds_id'])
if unknown:
    raise ValueError(f'DSSs in the LOINC check but not in DSS_View: {sorted(unknown)}')
loinc_source = loinc_readme[loinc_readme.str.startswith('LOINC source:')].iloc[0]
print(loinc_source)

LOINC_AXES = ['component', 'property', 'time', 'system', 'scale', 'method', 'class']
loinc_dss = loinc.groupby(['bc_id', 'ds_id']).agg(
    **{f'loinc_{x}': (f'loinc_{x}', lambda s: '; '.join(sorted(set(s)))) for x in LOINC_AXES + ['code']}).reset_index()
loinc_dss = loinc_dss.merge(view[['ds_id', 'TESTCD_value']], on='ds_id')

rows = []
for _, b in fanout[fanout['bc_id'].isin(loinc['bc_id'])].iterrows():
    p = loinc_dss[loinc_dss['bc_id'] == b['bc_id']]
    shared = loinc[loinc['bc_id'] == b['bc_id']].groupby('loinc_code')['ds_id'].nunique()
    differ = ', '.join(x for x in LOINC_AXES if p[f'loinc_{x}'].nunique() > 1) or '(none)'
    rows.append({
        'domain': b['domain'],
        'bc_id': b['bc_id'],
        'bc_short_name': b['bc_short_name'],
        'DSS_Count': b['DSS_Count'],
        'DSSs_With_LOINC': len(p),
        'COSMoS_Axes': b['Axes'],
        'LOINC_Axes_Differ': differ if len(p) > 1 else 'fewer than two DSSs with a LOINC code',
        'LOINC_Code_On_Several_DSSs': ', '.join(shared[shared > 1].index),
        'DSS_LOINC': ' | '.join(f"{r['ds_id']} ({r['TESTCD_value']}): {r['loinc_code']}" for _, r in p.iterrows()),
    })
loinc_compare = pd.DataFrame(rows)

fan_loinc_domains = fanout[fanout['domain'].isin(loinc['domain'])]
print(f'BCs that fan out in {", ".join(sorted(set(loinc["domain"])))}: {len(fan_loinc_domains)}   with a LOINC code: {len(loinc_compare)}   with two or more DSSs with a LOINC code: {int((loinc_compare["DSSs_With_LOINC"] > 1).sum())}')
in_list = lambda col, sep, x: loinc_compare[col].str.split(sep).apply(lambda a: x in a)
loinc_agreement = []
for cosmos_axis, loinc_axis in [('specimen', 'system'), ('method', 'method'), ('scale', 'scale')]:
    c, l = in_list('COSMoS_Axes', '; ', cosmos_axis), in_list('LOINC_Axes_Differ', ', ', loinc_axis)
    both, cosmos_only, loinc_only = int((c & l).sum()), int((c & ~l).sum()), int((~c & l).sum())
    loinc_agreement.append((f'{cosmos_axis}/{loinc_axis}', both, cosmos_only, loinc_only))
    print(f'{cosmos_axis} / {loinc_axis}: both {both}   COSMoS only {cosmos_only}   LOINC only {loinc_only}')
for x in ['component', 'property', 'time', 'class']:
    print(f'LOINC {x} differs: {int(in_list("LOINC_Axes_Differ", ", ", x).sum())} BCs')
print('LOINC code on several DSSs of one BC:', ', '.join(loinc_compare.loc[loinc_compare['LOINC_Code_On_Several_DSSs'] != '', 'bc_short_name']) or '(none)')

lb = view.loc[view['domain'] == 'LB', ['ds_id', 'CAT_value']].merge(loinc[['ds_id', 'loinc_class']].drop_duplicates(), on='ds_id')
lb['CAT_value'] = lb['CAT_value'].replace('', '(not assigned)')
lbcat_loinc_class = pd.crosstab(lb['CAT_value'], lb['loinc_class']).reset_index().rename(columns={'CAT_value': 'LBCAT'})
print(lbcat_loinc_class.to_string(index=False))

LOINC source: XML4Pharma LOINC web services (http://www.xml4pharmaserver.com:8080/LOINCService/rest/), LOINC 2.82, cache fetched 2026-08-24
BCs that fan out in LB, MB: 34   with a LOINC code: 30   with two or more DSSs with a LOINC code: 30
specimen / system: both 26   COSMoS only 1   LOINC only 1
method / method: both 4   COSMoS only 0   LOINC only 3
scale / scale: both 8   COSMoS only 0   LOINC only 1
LOINC component differs: 2 BCs
LOINC property differs: 13 BCs
LOINC time differs: 0 BCs
LOINC class differs: 11 BCs
LOINC code on several DSSs of one BC: Nicotine Measurement
         LBCAT  BLDBK  CELLMARK  CHEM  COAG  DRUG/TOX  HEM/BC  SPEC  UA
(not assigned)      1         0     7     0         0       0     0   0
     CHEMISTRY      0         0    61     1         5       0     0   1
    HEMATOLOGY      0         1     2     6         0      24     0   1
    URINALYSIS      0         0     9     0         0       1     1  16


## 4e. Who rates

Where the standards publish who completes or rates an instrument:

- SDTM CT: the instrument's class (QS, FT, RS) and the definitions of the class category
  codelists QSCAT, FTCAT, CCCAT; the Evaluator codelist EVAL.
- NCIt: the instrument definition (prose), and the clinical outcome assessment types under
  C142378 — are instruments classified under them (direct parent)?
- COSMoS: assigned values and value lists of `--EVAL` and `--EVALID` in QS, FT and RS.

`Rater_Evidence`: one row per instrument codelist in `SDTM_Instrument_Identity.xlsx`, with
the phrases in its NCIt definition that name who completes or rates it, quoted as found.
The phrase list is ours and is a reading aid; the phrases are not mapped to rater
categories. A shared classification from a published ontology is for later.
`Rater_COSMoS_EVAL`: the `--EVAL` / `--EVALID` values COSMoS assigns.
`Rater_Class_Moves`: instruments whose class category term (QSCAT, FTCAT, CCCAT) differs
between `CT_PRIOR_FILE` and the current CT, with the rater phrases in their NCIt definition.

In [8]:
instruments = pd.read_excel(INSTRUMENT_ID, sheet_name='Instruments', dtype=str).fillna('')
if INSTRUMENT_ID.stat().st_mtime < CT_FILE.stat().st_mtime:
    raise ValueError(f'{INSTRUMENT_ID.name} is older than {CT_FILE.name}; re-run SDTM_Instrument_Identity_Enrich')

# Phrases that name who completes or rates an instrument. Matched in the NCIt definition and quoted as found;
# not mapped to rater categories.
RATER_PATTERN = re.compile(
    r'(self[- ]report\w*|self[- ]administ\w*|patient[- ]report\w*|parent[- ]report\w*|(?:parent|observer|self|patient)[- ]rat\w*|observer[- ]report\w*'
    r'|proxy[- ]report\w*|completed by (?:the |a |an )?(?:health ?(?:care )?professional|\w+)|by (?:a |the )?clinician|clinician[- ]rat\w*'
    r'|clinician[- ]administ\w*|clinician[- ]report\w*|clinician[- ]determin\w*|clinician rates|caregiver\w*|physician\w*)',
    re.I)
instruments['Rater_Phrases'] = instruments['Instrument_NCIt_Definition'].apply(
    lambda s: '; '.join(sorted({m.lower() for m in RATER_PATTERN.findall(s)})))
rater_evidence = instruments[['Codelist_TESTCD', 'Instrument_Label', 'Domain', 'Instrument_NCIt_Code',
                              'Instrument_NCIt_Preferred_Term', 'Rater_Phrases', 'Instrument_NCIt_Definition']]

for sv in ['QSCAT', 'FTCAT', 'CCCAT']:
    print(f"{sv}: {codelists.loc[codelists['CDISC Submission Value'] == sv, 'CDISC Definition'].iloc[0]}")
domain_codelist = codelists.index[codelists['CDISC Submission Value'] == 'DOMAIN'][0]
for sv in ['QS', 'FT', 'RS', 'CC']:
    d = ct[(ct['Codelist Code'] == domain_codelist) & (ct['CDISC Submission Value'] == sv)].iloc[0]
    print(f"Domain {sv} ({d['Code']}): {d['CDISC Definition']}")

by_class = rater_evidence.groupby('Domain').agg(
    Instruments=('Codelist_TESTCD', 'size'),
    With_NCIt_Code=('Instrument_NCIt_Code', lambda s: int((s != '').sum())),
    With_Rater_Phrase=('Rater_Phrases', lambda s: int((s != '').sum())))
print(by_class.to_string())
phrases = rater_evidence.assign(phrase=rater_evidence['Rater_Phrases'].str.split('; ')).explode('phrase').reset_index(drop=True)
phrases = phrases[phrases['phrase'] != '']
print(pd.crosstab(phrases['phrase'], phrases['Domain']).to_string())

# NCIt clinical outcome assessment types: direct children of C142378. Are instruments classified under them?
csv.field_size_limit(10**9)
COA_PARENT = 'C142378'
names, parents_of, definitions = {}, {}, {}
with open(THESAURUS, encoding='utf-8', newline='') as f:
    for r in csv.reader(f, delimiter='\t'):
        names[r[0]] = r[3].split('|')[0]
        parents_of[r[0]] = set(r[2].split('|'))
        definitions[r[0]] = r[4]
coa_types = sorted(c for c, p in parents_of.items() if COA_PARENT in p)
instrument_codes = set(rater_evidence['Instrument_NCIt_Code']) - {''}
print(f'NCIt children of {COA_PARENT} {names[COA_PARENT]}:')
for code in coa_types:
    print(f'  {code} {names[code]}: {sum(code in parents_of[c] for c in instrument_codes)} instruments with it as direct parent')
instruments_under_coa = sum(bool(parents_of[c] & set(coa_types)) for c in instrument_codes)

eval_codelist = codelists.index[codelists['CDISC Submission Value'] == 'EVAL'][0]
print(f"SDTM CT {eval_codelist} EVAL: {codelists.loc[eval_codelist, 'CDISC Definition']} Terms: {int((ct['Codelist Code'] == eval_codelist).sum())}")
ev = variables[variables['variable_name'].isin(['QSEVAL', 'FTEVAL', 'RSEVAL', 'QSEVALID', 'FTEVALID', 'RSEVALID'])]
rater_cosmos_eval = (ev.assign(Assigned=ev['assigned_term_value'].replace('', '(value list only)'))
                     .groupby(['variable_name', 'Assigned'])
                     .agg(DSSs=('ds_id', 'nunique'), BCs=('bc_id', 'nunique'))
                     .reset_index())
print(rater_cosmos_eval.to_string(index=False))

# Instruments that changed class between a prior SDTM CT and the current one, read from the class category
# codelists QSCAT, FTCAT, CCCAT. Their term codes are NCIt instrument concepts. CT_PRIOR_FILE = None skips this.
rater_class_moves = pd.DataFrame()
if CT_PRIOR_FILE:
    prior_ct = pd.read_csv(CT_PRIOR_FILE, sep='\t', dtype=str).fillna('')
    class_of_codelist = {codelists.index[codelists['CDISC Submission Value'] == sv][0]: cls
                         for sv, cls in [('QSCAT', 'QS'), ('FTCAT', 'FT'), ('CCCAT', 'RS')]}

    def class_terms(c):
        t = c[c['Codelist Code'].isin(class_of_codelist)]
        return t.assign(Class=t['Codelist Code'].map(class_of_codelist)).set_index('Code')[['CDISC Submission Value', 'Class']]

    moves = class_terms(prior_ct).join(class_terms(ct), lsuffix='_Prior', rsuffix='_Current', how='inner')
    moves = moves[moves['Class_Prior'] != moves['Class_Current']].reset_index().rename(columns={'Code': 'NCIt_Code'})
    moves['NCIt_Preferred_Term'] = moves['NCIt_Code'].map(names)
    moves['Rater_Phrases'] = moves['NCIt_Code'].map(definitions).apply(
        lambda s: '; '.join(sorted({m.lower() for m in RATER_PATTERN.findall(s)})))
    moves['NCIt_Definition'] = moves['NCIt_Code'].map(definitions)
    rater_class_moves = moves
    print(f'Class moves, SDTM CT in {CT_PRIOR_FILE.parent.name} -> current: {len(rater_class_moves)}')
    print(rater_class_moves[['NCIt_Code', 'CDISC Submission Value_Current', 'Class_Prior', 'Class_Current', 'Rater_Phrases']].to_string(index=False))

QSCAT: A grouping of observations within the Questionnaires domain.
FTCAT: A grouping of observations within the Functional Tests domain.
CCCAT: A grouping of observations within the Disease Response and Clin Classification domain.
Domain QS (C49609): A findings domain that contains data for named, stand-alone instruments designed to provide an assessment of a concept. Questionnaires have a defined standard structure, format, and content; consist of conceptually related items that are typically scored; and have documented methods for administration and analysis.
Domain FT (C117756): A findings domain that contains data for named, stand-alone, task-based evaluations designed to provide an assessment of mobility, dexterity, or cognitive ability.
Domain RS (C107097): A findings domain for the assessment of disease response to therapy based on published criteria.
Domain CC (C228234): A findings domain that contains data for named instruments whose output is an ordinal or categorical score 

## 4f. Our position: `Variable_Meaning`

Sections 4b to 4e record what the standards publish (Layer A). `Variable_Meaning` in
`SDTM_Domain_Metadata.xlsx` is our position (Layer B): hand-kept, Findings only, one row per
variable or axis that bears on the observable question, each with a stated basis:
*sources agree*, *sources disagree, decided by a stated rule*, or *standards silent, open*.

This cell checks the sheet and stops if a row does not hold: the key must exist in the
sheet named by `Key_Source` (`Variable_Evidence`, `Key_Compare`, `Rater_COSMoS_EVAL` here, or
`Subject_State` of `COSMoS_Observable_LOINC_Check.xlsx` for `LOINC_Subject_State`), the domain must be Findings, the fixed lists must hold, and
`Position` is `open` exactly when `Basis` is *standards silent, open*. It does not judge
the positions.

In [9]:
MEANING_COLUMNS = ['Key_Source', 'Domain', 'Key', 'Position', 'Basis', 'Basis_Note', 'Status']
KEY_SOURCES = ['Variable_Evidence', 'Key_Compare', 'Rater_COSMoS_EVAL', 'LOINC_Subject_State']
POSITIONS = ['component', 'system', 'method', 'scale', 'not part of the observable', 'open']
BASES = ['sources agree', 'sources disagree, decided by a stated rule', 'standards silent, open']
STATUSES = ['proposed', 'accepted']

if list(meaning.columns) != MEANING_COLUMNS:
    raise ValueError(f'Variable_Meaning columns {list(meaning.columns)}, expected {MEANING_COLUMNS}')
for col, allowed in [('Key_Source', KEY_SOURCES), ('Position', POSITIONS), ('Basis', BASES), ('Status', STATUSES)]:
    bad = meaning[~meaning[col].isin(allowed)]
    if len(bad):
        raise ValueError(f'Variable_Meaning: {col} not in {allowed}:\n{bad[["Domain", "Key", col]].to_string(index=False)}')
bad = meaning[(meaning['Position'] == 'open') != (meaning['Basis'] == 'standards silent, open')]
if len(bad):
    raise ValueError(f'Variable_Meaning: Position open must go with Basis "standards silent, open":\n{bad[["Domain", "Key", "Position", "Basis"]].to_string(index=False)}')
bad = meaning[meaning['Basis_Note'] == '']
if len(bad):
    raise ValueError(f'Variable_Meaning: empty Basis_Note:\n{bad[["Domain", "Key"]].to_string(index=False)}')
bad = meaning[meaning.duplicated(['Key_Source', 'Domain', 'Key'], keep=False)]
if len(bad):
    raise ValueError(f'Variable_Meaning: duplicate keys:\n{bad[["Key_Source", "Domain", "Key"]].to_string(index=False)}')

findings_domains = set(meta.loc[meta['Observation_Class'] == 'Findings', 'Domain'])
evidence_keys = set(zip(variable_evidence['domain'], variable_evidence['variable']))
axis_keys = {a for col in ['Axes', 'Key_Differs_By'] for s in key_compare[col] for a in s.split('; ')} - {'', '(none)'}
eval_keys = set(rater_cosmos_eval['variable_name'])
state_keys = set(pd.read_excel(LOINC_CHECK, sheet_name='Subject_State', dtype=str)['state_variable'])


def key_problem(r):
    if r['Key_Source'] == 'Variable_Evidence':
        if (r['Domain'], r['Key']) not in evidence_keys:
            return 'not in Variable_Evidence'
    elif r['Key_Source'] == 'Key_Compare':
        if r['Domain'] != '':
            return 'Domain must be empty for Key_Compare'
        if r['Key'] not in axis_keys:
            return 'axis not in Key_Compare'
        return ''
    elif r['Key_Source'] == 'LOINC_Subject_State':
        if r['Key'] not in state_keys or not r['Key'].startswith(r['Domain']):
            return 'not in Subject_State for this domain'
    elif r['Key'] not in eval_keys or not r['Key'].startswith(r['Domain']):
        return 'not in Rater_COSMoS_EVAL for this domain'
    if r['Domain'] not in findings_domains:
        return 'domain not Findings'
    return ''


meaning['Problem'] = meaning.apply(key_problem, axis=1)
bad = meaning[meaning['Problem'] != '']
if len(bad):
    raise ValueError(f'Variable_Meaning: keys that do not hold:\n{bad[["Key_Source", "Domain", "Key", "Problem"]].to_string(index=False)}')
meaning = meaning.drop(columns='Problem')

print(f'Variable_Meaning: {len(meaning)} rows, all keys found')
print(meaning.groupby(['Basis', 'Position']).size().to_string())

Variable_Meaning: 23 rows, all keys found
Basis                                       Position                  
sources agree                               component                     4
                                            method                        3
                                            not part of the observable    2
                                            system                        3
sources disagree, decided by a stated rule  component                     2
                                            scale                         1
standards silent, open                      open                          8


## 4g. What the SDTMIG v3.4 Assumptions say

The SDTMIG domain Assumptions are prose, not in the variable tables. This section reads them from the IG PDF
for the domains of our variables (`Variable_Evidence` Findings rows, `Rater_COSMoS_EVAL`, and the
subject-state variables of the LOINC check). One row per numbered assumption that names one of our variables
or its `--` form. The text is quoted as pypdf extracts it: some words come out split (e.g. "identifie s"),
so variable names are matched on a copy with spaces between capitals removed. Nothing is classified here.

In [10]:
import pypdf

reader = pypdf.PdfReader(IG_PDF)
if reader.is_encrypted:
    reader.decrypt('')  # copying is allowed; AES needs the cryptography package
FURNITURE = re.compile(r'^(CDISC Study Data Tabulation Model Implementation Guide|© \d{4} Clinical Data Interchange|Page \d+$|\d{4}-\d{2}-\d{2}$)')
pdf_lines = [(p, l.strip()) for p, page in enumerate(reader.pages, start=1)
             for l in (page.extract_text() or '').split('\n') if l.strip() and not FURNITURE.match(l.strip())]
HEAD = re.compile(r'^([A-Z]{2}) – (.*Assumptions)$')
STOP = re.compile(r'^[A-Z]{2} – |^\d+\.\d+(\.\d+)* [A-Z]')
ITEM = re.compile(r'^(\d{1,2})\. (.*)')

our_variables = ({v for s in findings['variable'] for v in s.split('; ')}
                 | set(rater_cosmos_eval['variable_name'])
                 | set(pd.read_excel(LOINC_CHECK, sheet_name='Subject_State', dtype=str)['state_variable']))
scope = sorted({v[:2] for v in our_variables})
rows = []
for i, (p, l) in enumerate(pdf_lines):
    m = HEAD.match(l)
    if not m or m.group(1) not in scope:
        continue
    item = None
    for p2, l2 in pdf_lines[i + 1:]:
        if STOP.match(l2):
            break
        n = ITEM.match(l2)
        if n:
            item = {'Domain': m.group(1), 'Section': l, 'Item': int(n.group(1)), 'Page': p2, 'Text': n.group(2)}
            rows.append(item)
        elif item:
            item['Text'] += ' ' + l2
missing = sorted(set(scope) - {r['Domain'] for r in rows})
if missing:
    raise ValueError(f'No Assumptions items found in {IG_PDF.name} for: {missing}')


def named(r):
    joined = re.sub(r'(?<=[A-Z]) (?=[A-Z])', '', r['Text'])  # pypdf splits some words
    own = [v for v in sorted(our_variables) if v[:2] == r['Domain']]
    found = [v for v in own if re.search(r'\b' + v + r'\b', joined)]
    found += [f'--{v[2:]}' for v in own if re.search(r'--\s?' + v[2:] + r'\b', r['Text'])]
    return '; '.join(dict.fromkeys(found))


ig_assumptions = pd.DataFrame(rows)
ig_assumptions.insert(4, 'Variables_Named', ig_assumptions.apply(named, axis=1))
ig_assumptions = ig_assumptions[ig_assumptions['Variables_Named'] != ''].reset_index(drop=True)
print(f'SDTMIG v3.4 Assumptions in {", ".join(scope)}: {len(rows)} items; naming our variables: {len(ig_assumptions)}')
print(ig_assumptions[['Domain', 'Item', 'Page', 'Variables_Named']].to_string(index=False))

SDTMIG v3.4 Assumptions in GF, IS, LB, MB, MI, MK, QS, RS, TR, TU, VS: 98 items; naming our variables: 19
Domain  Item  Page          Variables_Named
    GF     8   224                 GFMETHOD
    IS     7   232                 ISBDAGNT
    IS     8   232                 ISTSTOPO
    LB     8   246 LBFAST; LBMETHOD; LBSPEC
    MB     1   251                 MBTSTDTL
    MI     2   265                 MITSTDTL
    QS     8   328                   --EVAL
    RS     1   332                    RSCAT
    RS     6   333                    RSCAT
    RS     7   333         RSEVAL; RSEVALID
    RS     9   334                    RSCAT
    RS     2   336                    RSCAT
    RS     1   337                    --CAT
    RS     6   338                    --CAT
    RS     8   338         --EVAL; --EVALID
    TU     8   348                 TUEVALID
    TR     1   352                  --GRPID
    TR     2   352                  --GRPID
    TR     6   353                   TREVAL


## 4h. The consumer key against the derivation's key

`Specimen_Findings` (LB, MB, MI) carries an observable key per DSS, built from our position in
`Variable_Meaning` (its section 4c): `bc_id | component | system | scale | method`, with scale read
as units present or absent. `COSMoS_Observable_Derivation` has its own key, with rules of its own
and scale read from the `--ORRES` data type. This section checks, for the DSSs in
`Specimen_Findings`, whether the two keys group the DSSs the same way, and lists every group where
they do not: a consumer key that joins DSSs the derivation keeps apart, or a derivation key that
the consumer key splits. It reports; it changes neither key.

The cell stops if `Specimen_Findings.xlsx` has no key, is older than `SDTM_Domain_Metadata.xlsx`
(the key would not reflect the current `Variable_Meaning`), has a DSS not in the derivation's
key, or carries a `package_date` that differs from `DSS_View` for the same DSS.


In [11]:
spec_ms = pd.read_excel(SPECIMEN_FINDINGS, sheet_name='Measurement_Specs', dtype=str).fillna('')
if 'Observable_Key' not in spec_ms.columns:
    raise ValueError('Specimen_Findings.xlsx has no Observable_Key - re-run Specimen_Findings first')
if SPECIMEN_FINDINGS.stat().st_mtime < DOMAIN_META.stat().st_mtime:
    raise ValueError('Specimen_Findings.xlsx is older than SDTM_Domain_Metadata.xlsx - re-run Specimen_Findings first')
dates = spec_ms[['ds_id', 'package_date']].merge(view[['ds_id', 'package_date']], on='ds_id', how='left', suffixes=('', '_view'))
bad = dates[dates['package_date'] != dates['package_date_view']]
if len(bad):
    raise ValueError(f'Specimen_Findings package_date differs from DSS_View:\n{bad.to_string(index=False)}')
ck = spec_ms[['ds_id', 'domain', 'Observable_Key']].merge(coord[['ds_id', 'observable_key']], on='ds_id', how='left')
bad = ck[ck['observable_key'].isna()]
if len(bad):
    raise ValueError(f'Specimen_Findings DSSs not in the derivation key:\n{bad[["domain", "ds_id"]].to_string(index=False)}')

rows = []
for mine, theirs, kind in [('Observable_Key', 'observable_key', 'consumer key joins DSSs the derivation keeps apart'),
                           ('observable_key', 'Observable_Key', 'consumer key splits a derivation key')]:
    for key, g in ck.groupby(mine):
        if g[theirs].nunique() > 1:
            rows.append({'Difference': kind, 'domain': '; '.join(sorted(set(g['domain']))), 'Key': key,
                         'DSSs': '; '.join(g['ds_id']), 'Other_Keys': ' || '.join(sorted(set(g[theirs])))})
consumer_key_compare = pd.DataFrame(rows, columns=['Difference', 'domain', 'Key', 'DSSs', 'Other_Keys'])

print(f'Specimen_Findings DSSs: {len(ck)}   consumer keys: {ck["Observable_Key"].nunique()}   derivation keys: {ck["observable_key"].nunique()}')
print(f'Groups that differ: {len(consumer_key_compare)}')
if len(consumer_key_compare):
    print(consumer_key_compare.to_string(index=False))


Specimen_Findings DSSs: 169   consumer keys: 167   derivation keys: 167
Groups that differ: 0


## 5. Domain profile

In [12]:
instrument_domains = set(instr_rules['Domain'])

def consumer_class(row):
    if row['Specimen_Based'] == 'Yes':
        return 'Specimen'
    if row['Measurement'] == 'Yes':
        return 'Measurement'
    if row['Domain'] in instrument_domains:
        return 'Instrument'
    return ''

meta['Consumer_Class'] = meta.apply(consumer_class, axis=1)
CONSUMER_OF = {'Specimen': 'Specimen_Findings', 'Measurement': 'Measurement_Findings', 'Instrument': 'Instrument_Findings'}
excluded = set(zip(excl['Consumer'], excl['Domain']))

def share(g, cols):
    cols = [c for c in cols if c in g.columns]
    return round(g[cols].ne('').any(axis=1).mean(), 2) if cols else 0.0

profile_rows = []
for domain, g in view.groupby('domain'):
    per_bc = g.groupby('bc_id').size()
    f = fanout[fanout['domain'] == domain]
    axis_counts = Counter(a for axes in f['Axes'] for a in axes.split('; '))
    top = max(axis_counts.values()) if axis_counts else 0
    primary = '; '.join(sorted(a for a, n in axis_counts.items() if n == top)) if axis_counts else ''
    scales = Counter(s for v in g['result_scales'] for s in v.split(';') if s)
    m = meta[meta['Domain'] == domain]
    cls = m['Consumer_Class'].iloc[0] if len(m) else ''
    if not len(m):
        status = 'not in domain metadata'
    elif not cls:
        status = 'no consumer class'
    elif (CONSUMER_OF[cls], domain) in excluded:
        status = f'excluded from {CONSUMER_OF[cls]}'
    else:
        status = f'in {CONSUMER_OF[cls]}'
    profile_rows.append({
        'domain': domain,
        'Observation_Class': m['Observation_Class'].iloc[0] if len(m) else '',
        'Consumer_Class': cls,
        'Consumer_Status': status,
        'BCs': per_bc.size,
        'DSSs': len(g),
        'BCs_Fanning_Out': int((per_bc > 1).sum()),
        'Max_Fanout': int(per_bc.max()),
        'Primary_Axis': primary,
        'Axes_Seen': '; '.join(f'{a} {n}' for a, n in sorted(axis_counts.items(), key=lambda kv: (-kv[1], kv[0]))),
        'Scale_Mix': '; '.join(f'{s} {n}' for s, n in sorted(scales.items(), key=lambda kv: (-kv[1], kv[0]))),
        'Share_With_Units': round(g['has_units'].mean(), 2),
        'Share_With_Specimen': share(g, AXIS_COLUMNS['specimen']),
        'Share_With_Method': share(g, AXIS_COLUMNS['method']),
        'Share_With_Location': share(g, AXIS_COLUMNS['location']),
    })
profile = pd.DataFrame(profile_rows)
with pd.option_context('display.width', 220, 'display.max_columns', None, 'display.max_rows', None):
    print(profile[['domain', 'Consumer_Status', 'BCs', 'DSSs', 'BCs_Fanning_Out', 'Max_Fanout', 'Primary_Axis']].to_string(index=False))

domain                 Consumer_Status  BCs  DSSs  BCs_Fanning_Out  Max_Fanout       Primary_Axis
    AE               no consumer class    1     2                1           2             (none)
    BE               no consumer class    7     7                0           1                   
    CM               no consumer class    1     4                1           4           category
    DD               no consumer class   13    13                0           1                   
    DM               no consumer class    5     5                0           1                   
    DS               no consumer class   20    20                0           1                   
    EC               no consumer class    1     2                1           2             (none)
    EG         in Measurement_Findings   33    33                0           1                   
    EX               no consumer class    1     1                0           1                   
    FA              

## 6. Scale and units

Grain: domain × the result-scale combination COSMoS publishes on the BC. Flags only for
single-scale combinations: Quantitative without units, and Nominal or Ordinal with units.

In [13]:
su = view.groupby(['domain', 'result_scales']).agg(DSSs=('ds_id', 'size'), With_Units=('has_units', 'sum')).reset_index()
su['Without_Units'] = su['DSSs'] - su['With_Units']

def su_flag(r):
    if r['result_scales'] == 'Quantitative' and r['Without_Units'] > 0:
        return 'Quantitative without units'
    if r['result_scales'] in ('Nominal', 'Ordinal') and r['With_Units'] > 0:
        return f'{r["result_scales"]} with units'
    return ''

su['Flag'] = su.apply(su_flag, axis=1)
scale_units = su.rename(columns={'result_scales': 'Result_Scales'})
print(scale_units[scale_units['Flag'] != ''].to_string(index=False))

domain Result_Scales  DSSs  With_Units  Without_Units                       Flag
    DM  Quantitative     1           0              1 Quantitative without units
    LB  Quantitative    76          72              4 Quantitative without units
    MB  Quantitative     2           1              1 Quantitative without units
    MI  Quantitative     1           0              1 Quantitative without units
    MK  Quantitative    25          21              4 Quantitative without units
    RE  Quantitative   125         110             15 Quantitative without units
    RP  Quantitative    37          10             27 Quantitative without units
    RS       Ordinal   124          19            105         Ordinal with units
    SC  Quantitative    10           2              8 Quantitative without units
    SR  Quantitative     6           5              1 Quantitative without units
    TS  Quantitative     8           0              8 Quantitative without units
    UR  Quantitative     5  

## 7. Exclusion claims

Each row of `Consumer_Exclusions` states one claim in machine form (`Check`):

- `no_fanout` — no BC in the domain has more than one DSS
- `no_axis` — no BC that fans out differs by any axis variable
- `not_axis:<axis>` — no BC that fans out differs by `<axis>`
- `axis_all:<axis>` — every BC that fans out differs by `<axis>`
- `not_primary:<axis>` — `<axis>` is not the primary axis (the axis most BCs that fan out differ by)

In [14]:
def evaluate(domain, check):
    f = fanout[fanout['domain'] == domain]
    n = len(f)
    if not (view['domain'] == domain).any():
        return 'no content', 'no DSSs in this release'
    if check == 'no_fanout':
        return ('yes' if n == 0 else 'no'), f'{n} BCs fan out'
    axis_sets = [set(a.split('; ')) - {'(none)'} for a in f['Axes']]
    seen = Counter(a for s in axis_sets for a in s)
    evidence = f'{n} BCs fan out; axes: ' + ('; '.join(f'{a} {c}' for a, c in sorted(seen.items(), key=lambda kv: (-kv[1], kv[0]))) or 'none')
    if check == 'no_axis':
        return ('yes' if not seen else 'no'), evidence
    kind, _, axis = check.partition(':')
    if axis not in AXIS_COLUMNS and axis != 'scale':
        raise ValueError(f'Unknown axis in check {check!r}')
    if kind == 'not_axis':
        return ('yes' if seen[axis] == 0 else 'no'), evidence
    if kind == 'axis_all':
        return ('yes' if n and seen[axis] == n else 'no'), evidence
    if kind == 'not_primary':
        top = max(seen.values()) if seen else 0
        return ('yes' if seen[axis] < top else 'no'), evidence
    raise ValueError(f'Unknown check {check!r}')

claims = excl.copy()
claims[['Holds', 'Evidence']] = claims.apply(lambda r: pd.Series(evaluate(r['Domain'], r['Check'])), axis=1)
claims = claims[['Consumer', 'Domain', 'Claim_ID', 'Claim', 'Check', 'Holds', 'Evidence', 'Reason']]
with pd.option_context('display.width', 220, 'display.max_colwidth', 90):
    print(claims[['Claim_ID', 'Claim', 'Holds', 'Evidence']].to_string(index=False))

Claim_ID                                           Claim Holds                                                                           Evidence
    IS-1 Every BC that fans out differs by binding agent   yes 6 BCs fan out; axes: binding agent 6; scale 4; test detail 2; method 1; specimen 1
    IS-2                Specimen is not the primary axis   yes 6 BCs fan out; axes: binding agent 6; scale 4; test detail 2; method 1; specimen 1
    GF-1   Every BC that fans out differs by test detail   yes                10 BCs fan out; axes: test detail 10; method 4; scale 4; specimen 1
    GF-2                Specimen is not the primary axis   yes                10 BCs fan out; axes: test detail 10; method 4; scale 4; specimen 1
    UR-1                     No BC has more than one DSS   yes                                                                      0 BCs fan out


## 8. Content without a consumer class

In [15]:
unclassified = profile[profile['Consumer_Status'].isin(['no consumer class', 'not in domain metadata'])].reset_index(drop=True)
print(', '.join(unclassified['domain']))

AE, BE, CM, DD, DM, DS, EC, EX, FA, IE, MH, PR, RP, SC, SR, SU, TR, TS, TU


## 9. Key measures and write

In [16]:
measures = [
    ('package_date', package_date),
    ('domains_with_dss', profile.shape[0]),
    ('dss', len(view)),
    ('bcs_with_dss', view['bc_id'].nunique()),
    ('bcs_fanning_out', len(fanout)),
    ('claims', len(claims)),
    ('claims_not_holding', int((claims['Holds'] == 'no').sum())),
    ('scale_unit_flags', int((scale_units['Flag'] != '').sum())),
    ('domains_with_dss_no_consumer_class', len(unclassified)),
    ('bcs_fanning_out_in_key_scope', len(in_scope)),
    ('variable_domain_pairs', len(variable_evidence)),
    ('variable_domain_pairs_assigned_differs', n_assigned),
    ('generic_variables_published_differently', len(differ)),
    ('variable_domain_pairs_findings', len(findings)),
    ('variable_domain_pairs_findings_assigned_differs', n_assigned_findings),
    ('variable_domain_pairs_findings_not_in_sdtmig', int((findings['IG_Role'] == '(not in SDTMIG v3.4)').sum())),
    ('variable_domain_pairs_findings_ig_role_differs', int((findings['IG_Role_Differs_From_Model'] == 'yes').sum())),
    ('bcs_fanning_out_with_loinc', len(loinc_compare)),
    ('instruments', len(rater_evidence)),
    ('instruments_with_rater_phrase', int((rater_evidence['Rater_Phrases'] != '').sum())),
    ('instruments_under_coa_type', instruments_under_coa),
    ('instrument_class_moves', len(rater_class_moves)),
    ('variable_meaning_rows', len(meaning)),
    ('ig_assumption_items_naming_our_variables', len(ig_assumptions)),
    ('consumer_key_dss', len(ck)),
    ('consumer_keys', ck['Observable_Key'].nunique()),
    ('consumer_key_groups_differing', len(consumer_key_compare)),
]
for _, r in profile.iterrows():
    measures += [
        (f'dss:{r["domain"]}', r['DSSs']),
        (f'bcs_fanning_out:{r["domain"]}', r['BCs_Fanning_Out']),
        (f'primary_axis:{r["domain"]}', r['Primary_Axis']),
    ]
for _, r in claims.iterrows():
    measures.append((f'claim:{r["Claim_ID"]}', r['Holds']))
for label, both, fanout_only, key_only in key_agreement:
    measures += [(f'key_both:{label}', both), (f'key_fanout_only:{label}', fanout_only), (f'key_only:{label}', key_only)]
for label, n in key_grain.items():
    measures.append((f'key_grain:{label}', n))
for label, both, cosmos_only, loinc_only in loinc_agreement:
    measures += [(f'loinc_both:{label}', both), (f'loinc_cosmos_only:{label}', cosmos_only), (f'loinc_only:{label}', loinc_only)]
for basis, n in meaning['Basis'].value_counts().sort_index().items():
    measures.append((f'variable_meaning:{basis}', n))
key_measures = pd.DataFrame(measures, columns=['measure', 'value']).astype({'value': str})

pins = pd.DataFrame([
    ('COSMoS package_date (latest in DSS sheet)', package_date),
    ('COSMoS_Graph.xlsx modified', pd.Timestamp(GRAPH.stat().st_mtime, unit='s').strftime('%Y-%m-%d %H:%M')),
    ('DSS_View.xlsx modified', pd.Timestamp(DSS_VIEW.stat().st_mtime, unit='s').strftime('%Y-%m-%d %H:%M')),
    ('SDTM_Domain_Metadata.xlsx modified', pd.Timestamp(DOMAIN_META.stat().st_mtime, unit='s').strftime('%Y-%m-%d %H:%M')),
    ('COSMoS_Observable_Derivation.xlsx modified', pd.Timestamp(OBS_KEY.stat().st_mtime, unit='s').strftime('%Y-%m-%d %H:%M')),
    ('COSMoS_Observable_LOINC_Check.xlsx', f'{loinc_generated}; {loinc_source}'),
    ('Specimen_Findings.xlsx modified', pd.Timestamp(SPECIMEN_FINDINGS.stat().st_mtime, unit='s').strftime('%Y-%m-%d %H:%M')),
    ('SDTM_Terminology.txt modified', pd.Timestamp(CT_FILE.stat().st_mtime, unit='s').strftime('%Y-%m-%d %H:%M')),
    ('SDTM_Instrument_Identity.xlsx modified', pd.Timestamp(INSTRUMENT_ID.stat().st_mtime, unit='s').strftime('%Y-%m-%d %H:%M')),
    ('Thesaurus.txt modified', pd.Timestamp(THESAURUS.stat().st_mtime, unit='s').strftime('%Y-%m-%d %H:%M')),
    ('SDTM_v2.0.csv modified', pd.Timestamp(SDTM_MODEL.stat().st_mtime, unit='s').strftime('%Y-%m-%d %H:%M')),
    ('SDTMIG_v3.4.csv modified', pd.Timestamp(SDTMIG.stat().st_mtime, unit='s').strftime('%Y-%m-%d %H:%M')),
    ('NSV registry', NSV_REGISTRY.name),
    ('SDTMIG PDF', IG_PDF.name),
    ('SDTM CT prior for class moves', str(CT_PRIOR_FILE.relative_to(REPO)) if CT_PRIOR_FILE else 'none'),
], columns=['pin', 'value'])

with pd.ExcelWriter(OUT, engine='openpyxl') as writer:
    profile.to_excel(writer, sheet_name='Domain_Profile', index=False)
    fanout.to_excel(writer, sheet_name='Fanout_Axes', index=False)
    key_compare.to_excel(writer, sheet_name='Key_Compare', index=False)
    variable_evidence.to_excel(writer, sheet_name='Variable_Evidence', index=False)
    consumer_key_compare.to_excel(writer, sheet_name='Consumer_Key_Compare', index=False)
    ig_assumptions.to_excel(writer, sheet_name='IG_Assumptions', index=False)
    loinc_compare.to_excel(writer, sheet_name='LOINC_Compare', index=False)
    lbcat_loinc_class.to_excel(writer, sheet_name='LBCAT_LOINC_Class', index=False)
    rater_evidence.to_excel(writer, sheet_name='Rater_Evidence', index=False)
    rater_cosmos_eval.to_excel(writer, sheet_name='Rater_COSMoS_EVAL', index=False)
    if len(rater_class_moves):
        rater_class_moves.to_excel(writer, sheet_name='Rater_Class_Moves', index=False)
    scale_units.to_excel(writer, sheet_name='Scale_Units', index=False)
    claims.to_excel(writer, sheet_name='Exclusion_Claims', index=False)
    unclassified.to_excel(writer, sheet_name='Unclassified_Content', index=False)
    key_measures.to_excel(writer, sheet_name='Key_Measures', index=False)
    pins.to_excel(writer, sheet_name='Pins', index=False)
print('written:', OUT)

written: /Users/kerstinforsberg/repos/GitHub/cdisc-for-ai/domain-behaviour/interim/Domain_Behaviour.xlsx


## 10. Release diff (set `PRIOR_FILE` above)

In [17]:
if PRIOR_FILE and Path(PRIOR_FILE).exists():
    prior = pd.read_excel(PRIOR_FILE, sheet_name='Key_Measures', dtype=str)
    prev = dict(zip(prior['measure'], prior['value']))
    curr = dict(zip(key_measures['measure'], key_measures['value']))
    print(f'Prior: {PRIOR_FILE}')
    print('Added measures:  ', sorted(set(curr) - set(prev)))
    print('Dropped measures:', sorted(set(prev) - set(curr)))
    changed = [(m, prev[m], curr[m]) for m in sorted(set(prev) & set(curr)) if prev[m] != curr[m]]
    print('Value changes (measure: prior -> current):')
    for m, a, b in changed:
        print(f'  {m}: {a} -> {b}')
    if not changed:
        print('  (none)')
else:
    print('PRIOR_FILE not set or missing - skipping release diff.')

PRIOR_FILE not set or missing - skipping release diff.


## 11. Publish as HTML

Writes `docs/analyses/domain-behaviour.html`, served by GitHub Pages and linked from the
landing page. Regenerated on every run, so the page always shows the last run's release.

In [18]:
import html
from datetime import date

HTML_OUT = REPO / 'docs' / 'analyses' / 'domain-behaviour.html'
HTML_OUT.parent.mkdir(exist_ok=True)
GH = 'https://github.com/kerfors/cdisc-for-ai'
e = lambda s: html.escape(str(s))

def table(df, cols, headers, cls=''):
    head = ''.join(f'<th>{e(h)}</th>' for h in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{v}</td>' for v in row) + '</tr>' for row in df[cols].itertuples(index=False))
    return f'<div class="scroll"><table class="{cls}"><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'

# Exclusion claims
c = claims.astype(str).copy()
c['Holds'] = c['Holds'].map(lambda h: f'<span class="{"ok" if h == "yes" else "no" if h == "no" else "na"}">{e(h)}</span>')
c['Claim'] = c.apply(lambda r: f'{e(r["Claim"])}<span class="desc">{e(r["Reason"])}</span>', axis=1)
for col in ['Claim_ID', 'Domain', 'Evidence']:
    c[col] = c[col].map(e)
claims_html = table(c, ['Claim_ID', 'Domain', 'Claim', 'Holds', 'Evidence'], ['Claim', 'Domain', 'Statement and reason', 'Holds', 'Evidence'])

# Domains: those with a consumer class or fan-out in the table, the rest folded
ORDER = ['in Specimen_Findings', 'excluded from Specimen_Findings', 'in Measurement_Findings',
         'in Instrument_Findings', 'no consumer class', 'not in domain metadata']
p = profile.copy()
p['_o'] = p['Consumer_Status'].map(lambda s: ORDER.index(s) if s in ORDER else 99)
p = p.sort_values(['_o', 'domain'])
main = p[(p['Consumer_Class'] != '') | (p['BCs_Fanning_Out'] > 0)].astype(str)
rest = p[(p['Consumer_Class'] == '') & (p['BCs_Fanning_Out'] == 0)].astype(str)
for d in (main, rest):
    for col in d.columns:
        d[col] = d[col].map(e)
PROFILE_COLS = ['domain', 'Consumer_Status', 'BCs', 'DSSs', 'BCs_Fanning_Out', 'Max_Fanout', 'Primary_Axis', 'Scale_Mix']
PROFILE_HEAD = ['Domain', 'Consumer', 'BCs', 'DSSs', 'BCs that fan out', 'Max fan-out', 'Primary axis', 'Result scales (BC)']
profile_html = table(main, PROFILE_COLS, PROFILE_HEAD, 'num')
if len(rest):
    profile_html += (f'<details><summary>{len(rest)} more domains with no consumer class and no fan-out: '
                     f'{e(", ".join(rest["domain"]))}</summary>' + table(rest, PROFILE_COLS, PROFILE_HEAD, 'num') + '</details>')

# Fan-out axes per domain, collapsible
blocks = []
for dom, g in fanout.groupby('domain'):
    g = g.astype(str).copy()
    for col in g.columns:
        g[col] = g[col].map(e)
    blocks.append(f'<details><summary><strong>{e(dom)}</strong> · {len(g)} BC{"s" if len(g) != 1 else ""} fan{"" if len(g) != 1 else "s"} out</summary>'
                  + table(g, ['bc_id', 'bc_short_name', 'DSS_Count', 'Axes', 'Axes_Value_List_Only'], ['BC', 'Name', 'DSSs', 'Axes that differ', 'Only the value list differs']) + '</details>')
fanout_html = '<div class="grid">' + '\n'.join(blocks) + '</div>'

su = scale_units[scale_units['Flag'] != ''].astype(str).copy()
for col in su.columns:
    su[col] = su[col].map(e)
su_html = table(su, ['domain', 'Result_Scales', 'DSSs', 'With_Units', 'Without_Units', 'Flag'],
                ['Domain', 'Result scale (BC)', 'DSSs', 'With units', 'Without units', 'Flag'], 'num3')

page = f'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Domain behaviour</title>
<style>
    :root {{ --text:#1a1a1a; --muted:#555; --rule:#e5e5e5; --link:#0b5fa5; --bg:#fdfdfd; --ok:#2e7d32; --no:#b3261e; }}
    html {{ box-sizing:border-box; }} *, *::before, *::after {{ box-sizing:inherit; }}
    body {{ font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Helvetica,Arial,sans-serif; color:var(--text); background:var(--bg);
           line-height:1.55; max-width:60rem; margin:3rem auto; padding:0 1rem; font-size:16px; }}
    h1 {{ font-size:1.75rem; font-weight:600; margin:0 0 .5rem; letter-spacing:-.01em; }}
    h2 {{ font-size:1.15rem; font-weight:600; margin:2.5rem 0 .75rem; padding-bottom:.35rem; border-bottom:1px solid var(--rule); }}
    p {{ margin:.75rem 0; max-width:42rem; }}
    a {{ color:var(--link); text-decoration:none; }} a:hover {{ text-decoration:underline; }}
    code {{ font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace; font-size:.9em; background:#f3f3f3; padding:.1em .35em; border-radius:3px; }}
    .lead {{ color:var(--muted); font-size:1.05rem; margin:-.25rem 0 1.5rem; line-height:1.4; }}
    .pins {{ color:var(--muted); font-size:.92em; }}
    .scroll {{ overflow-x:auto; margin:.5rem 0 1rem; }}
    table {{ border-collapse:collapse; width:100%; font-size:.9em; }}
    th, td {{ text-align:left; vertical-align:top; padding:.4rem .6rem; border-bottom:1px solid var(--rule); }}
    th {{ font-weight:600; white-space:nowrap; }}
    table.num td:nth-child(n+3):nth-child(-n+6), table.num3 td:nth-child(n+3):nth-child(-n+5) {{ text-align:right; font-variant-numeric:tabular-nums; }}
    .desc {{ display:block; color:var(--muted); font-size:.92em; margin-top:.15rem; }}
    .ok {{ color:var(--ok); font-weight:600; }} .no {{ color:var(--no); font-weight:600; }} .na {{ color:var(--muted); }}
    details {{ margin:.4rem 0; }} summary {{ cursor:pointer; padding:.25rem 0; }}
    .grid {{ display:grid; grid-template-columns:repeat(auto-fill, minmax(17rem, 1fr)); column-gap:1.5rem; }}
    .grid details[open] {{ grid-column:1 / -1; }}
    footer {{ margin-top:3rem; padding-top:1rem; border-top:1px solid var(--rule); color:var(--muted); font-size:.9em; }}
</style>
</head>
<body>
<p class="pins"><a href="../index.html">CDISC for AI</a> · Analyses</p>
<h1>Domain behaviour</h1>
<p class="lead">How each SDTM domain behaves in the published COSMoS content: how Biomedical Concepts fan out into Dataset Specializations, and which variables make the difference.</p>

<p>One schema serves every COSMoS domain, but the relationship between a Biomedical Concept (BC) and its Dataset Specializations (DSSs) behaves differently from domain to domain. This page re-derives that behaviour from each COSMoS release. An <em>axis</em> is a variable whose value differs between the DSSs of one BC: specimen, method, binding agent, test detail, location, category, evaluator. An axis is counted when the assigned value differs; where only the value list differs, it is shown in a separate column. <em>Scale</em> is read from units: some DSSs of the BC carry units, others do not.</p>
<p>The page reports; it does not decide. Which domains each Findings consumer covers is recorded in <a href="{GH}/blob/main/sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx"><code>SDTM_Domain_Metadata.xlsx</code></a>, and the reasons for leaving a domain out are stated there as claims. The first table tests those claims against this release.</p>
<p class="pins">COSMoS package {e(package_date)} · generated {date.today().isoformat()} · data: <a href="{GH}/raw/main/domain-behaviour/interim/Domain_Behaviour.xlsx"><code>Domain_Behaviour.xlsx</code></a> · notebook: <a href="{GH}/blob/main/domain-behaviour/notebooks/Domain_Behaviour.ipynb"><code>Domain_Behaviour.ipynb</code></a></p>

<h2>Exclusion claims</h2>
<p>Each reason for leaving a domain out of a consumer, stated as something the published content must show.</p>
{claims_html}

<h2>Domains</h2>
<p>One row per domain with Dataset Specializations. The primary axis is the axis most BCs that fan out differ by.</p>
{profile_html}

<h2>Where the DSSs of one BC differ</h2>
{fanout_html}

<h2>Scale and units</h2>
<details><summary>{len(su)} result-scale combinations where units and scale do not line up</summary>
<p>Quantitative without units, or Nominal and Ordinal with units. Reported, not judged — an ordinal score item can collect a raw value with units.</p>
{su_html}
</details>

<footer>
Part of <a href="../index.html">CDISC for AI</a>. The March 2026 analysis this page continues is archived in <a href="{GH}/tree/main/docs/archive/behavioural-analysis-2026-03"><code>docs/archive/behavioural-analysis-2026-03/</code></a>. Exploratory work, not an official CDISC product.
</footer>
</body>
</html>
'''
HTML_OUT.write_text(page, encoding='utf-8')
print('written:', HTML_OUT)

written: /Users/kerstinforsberg/repos/GitHub/cdisc-for-ai/docs/analyses/domain-behaviour.html
